# wandb run explorer

Filter, group, and inspect the active-learning runs logged to wandb by this project
(`utils.wandb_utils`) — learning curves (`test_acc`, `test_auroc`, ...) and the
`aubc_*` budget-curve summaries.

**Run naming this notebook relies on** (see `src/utils/wandb_utils.py`):
- run **name**  = `{data_name}/{method_variant}/seed-{seed}`
- run **group** = `{data_name}/{active_name}/{method_variant}`
- per-iteration history keys: `al/n_labelled` (x-axis) vs `al/test_acc`, `al/test_auroc`, ...
- summary keys: `al_summary/aubc_test_acc`, `al_summary/aubc_test_auroc`, ... (→ `aubc_*` columns here)

**Two data sources** (set `SOURCE` in the config cell):
- `"wandb"` — live, via `wandb.Api()` (needs network + `wandb login`).
- a **snapshot dir** (e.g. `analysis/wandb_snapshots/<date>`) written by
  `analysis/snapshot_wandb.py` — fully offline & reproducible.

Workflow: **load → filter → group/AUBC table → plot curves → AUBC bars**.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #
WANDB_PROJECT = "jingwusince01-national-taiwan-uni/realistic-al"

# Where to read runs from:
#   "wandb"                                  -> live API (needs network + `wandb login`)
#   "analysis/wandb_snapshots/<dir>"         -> offline snapshot (snapshot_wandb.py)
SOURCE = "wandb"
# SOURCE = "analysis/wandb_snapshots/2026-09-21_pre-pheno-backfill"  # <- offline fallback

# Optional server-side narrowing for the live API (MongoDB-style), None = all runs.
# Keys: config.<name>, summary_metrics.<name>, tags, state, display_name ($regex).
# Example: {"state": "finished", "config.data_name": "cifar10_imb"}
RUN_FILTERS = None

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.dpi"] = 110

In [ ]:
# --------------------------------------------------------------------------- #
# Loading: build one tidy row per run + fetch per-iteration history.
# Same interface for the live API and an offline snapshot (dispatch on SOURCE).
# --------------------------------------------------------------------------- #

def _seed_from_name(name):
    m = re.search(r"seed-(\d+)", name or "")
    return int(m.group(1)) if m else None

def _method_from(name, group):
    """The method-variant tag, e.g. 'vendi-q2-feat-cos-zscore-q2.0'."""
    if name and name.count("/") >= 2:
        return name.split("/")[1]          # data / METHOD / seed-N
    if group and group.count("/") >= 2:
        return group.split("/")[-1]        # data / active / METHOD
    return group or name

def _first(cfg, *keys):
    for k in keys:
        if k in cfg and cfg[k] not in (None, ""):
            return cfg[k]
    return None

def _run_row(rid, name, group, state, created, tags, config, summary, url):
    """Flatten one run into a tidy row (identity + key config + aubc summaries)."""
    cfg = config or {}
    active = _first(cfg, "active_name")
    if active is None and group and group.count("/") >= 2:
        active = group.split("/")[1]
    row = {
        "id": rid, "name": name, "group": group, "state": state,
        "created_at": created, "url": url, "tags": list(tags or []),
        # flat convenience aliases (init_wandb), falling back to nested config
        "data_name":    _first(cfg, "data_name", "data/name"),
        "active_name":  active,
        "query_name":   _first(cfg, "query_name", "query/name"),
        "model_name":   _first(cfg, "model_name", "model/name"),
        "method":       _method_from(name, group),
        "seed":         _first(cfg, "seed", "trainer/seed") or _seed_from_name(name),
        "num_labelled": _first(cfg, "num_labelled", "active/num_labelled"),
        "acq_size":     _first(cfg, "acq_size", "active/acq_size"),
        "num_iter":     _first(cfg, "num_iter", "active/num_iter"),
    }
    for k, v in (summary or {}).items():        # al_summary/aubc_test_acc -> aubc_test_acc
        if isinstance(k, str) and k.startswith("al_summary/aubc_"):
            row[k.replace("al_summary/aubc_", "aubc_")] = v
    return row

# ---- live wandb -------------------------------------------------------------
def _runs_from_wandb(project=WANDB_PROJECT, filters=RUN_FILTERS):
    import wandb
    api = wandb.Api(timeout=60)
    runs = api.runs(project, filters=filters, order="-created_at", per_page=500)
    rows = [
        _run_row(r.id, r.name, r.group, r.state, str(r.created_at),
                 list(r.tags), dict(r.config), dict(r.summary), r.url)
        for r in runs
    ]
    return pd.DataFrame(rows)

def _history_from_wandb(runs_df, metrics, x="n_labelled", project=WANDB_PROJECT):
    import wandb
    api = wandb.Api(timeout=60)
    want = ["al_iter", f"al/{x}"] + [f"al/{m}" for m in metrics]
    out = []
    for _, r in runs_df.iterrows():
        h = api.run(f"{project}/{r['id']}").history(samples=50000, pandas=True)
        if h is None or h.empty or f"al/{x}" not in h.columns:
            continue
        h = h[[c for c in want if c in h.columns]].dropna(subset=[f"al/{x}"]).copy()
        for meta in ["id", "name", "method", "data_name", "active_name", "query_name", "seed"]:
            h[meta] = r[meta]
        out.append(h)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

# ---- offline snapshot -------------------------------------------------------
def _runs_from_snapshot(snap_dir):
    df = pd.read_csv(Path(snap_dir) / "runs.csv")
    def _j(x, default):
        try:    return json.loads(x) if isinstance(x, str) else default
        except Exception: return default
    rows = [
        _run_row(r["id"], r.get("name"), r.get("group"), r.get("state"),
                 r.get("created_at"), _j(r.get("tags"), []),
                 _j(r.get("config"), {}), _j(r.get("summary"), {}), r.get("url"))
        for _, r in df.iterrows()
    ]
    return pd.DataFrame(rows)

def _history_from_snapshot(snap_dir, runs_df, metrics, x="n_labelled"):
    want = ["al_iter", f"al/{x}"] + [f"al/{m}" for m in metrics]
    out = []
    for _, r in runs_df.iterrows():
        p = Path(snap_dir) / "history" / f"{r['id']}.csv"
        if not p.exists():
            continue
        h = pd.read_csv(p)
        if f"al/{x}" not in h.columns:
            continue
        h = h[[c for c in want if c in h.columns]].dropna(subset=[f"al/{x}"]).copy()
        for meta in ["id", "name", "method", "data_name", "active_name", "query_name", "seed"]:
            h[meta] = r[meta]
        out.append(h)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

# ---- unified entry points ---------------------------------------------------
def load_runs():
    """One tidy row per run, from `SOURCE` (live wandb or a snapshot dir)."""
    return _runs_from_wandb() if SOURCE == "wandb" else _runs_from_snapshot(SOURCE)

def load_history(runs_df, metrics=("test_acc",), x="n_labelled"):
    """Long-form per-iteration history for the given runs (one row per al_iter)."""
    metrics = [metrics] if isinstance(metrics, str) else list(metrics)
    if SOURCE == "wandb":
        return _history_from_wandb(runs_df, metrics, x=x)
    return _history_from_snapshot(SOURCE, runs_df, metrics, x=x)

In [ ]:
# --------------------------------------------------------------------------- #
# Filter, group/summarize, and plot.
# --------------------------------------------------------------------------- #

def overview(runs):
    """Print what's in the loaded runs table so you know what to filter on."""
    print(f"{len(runs)} runs   states: {dict(runs['state'].value_counts())}")
    for col in ["data_name", "active_name", "query_name", "model_name"]:
        vc = runs[col].dropna().value_counts()
        print(f"\n{col}  ({vc.size} distinct):")
        for k, v in vc.items():
            print(f"    {str(k):30s} {v}")
    aubc = sorted(c.replace("aubc_", "") for c in runs.columns if c.startswith("aubc_"))
    print("\naubc_* metrics available:", aubc)

def filter_runs(df, *, name_contains=None, method_contains=None,
                tags_all=None, tags_any=None, created_after=None, **equals):
    """Filter runs. `equals` takes any column (data_name=, active_name=, query_name=,
    method=, state=, seed=, ...) as a scalar or a list/tuple of allowed values."""
    out = df
    for col, val in equals.items():
        if val is None:
            continue
        vals = list(val) if isinstance(val, (list, tuple, set)) else [val]
        out = out[out[col].isin(vals)]
    if name_contains:
        out = out[out["name"].str.contains(name_contains, na=False)]
    if method_contains:
        out = out[out["method"].str.contains(method_contains, na=False)]
    if tags_all:
        out = out[out["tags"].apply(lambda ts: all(t in ts for t in tags_all))]
    if tags_any:
        out = out[out["tags"].apply(lambda ts: any(t in ts for t in tags_any))]
    if created_after is not None:
        out = out[pd.to_datetime(out["created_at"], errors="coerce")
                  >= pd.to_datetime(created_after)]
    return out.copy()

def aubc_table(df, metric="test_acc", by=("data_name", "active_name", "method")):
    """Mean / std / n of `aubc_<metric>` across seeds, grouped by `by`,
    sorted best-first. Filter to one dataset+regime first for a clean ranking."""
    col, by = f"aubc_{metric}", list(by)
    if col not in df.columns:
        avail = [c.replace("aubc_", "") for c in df.columns if c.startswith("aubc_")]
        raise KeyError(f"{col!r} not found. Available metrics: {avail}")
    return (df.dropna(subset=[col]).groupby(by)[col]
              .agg(mean="mean", std="std", n="count").reset_index()
              .sort_values("mean", ascending=False))

def plot_curves(hist, metric="test_acc", x="n_labelled", by="method",
                agg_seeds=True, ax=None, figsize=(8, 5), title=None):
    """Learning curves: `al/<metric>` vs `al/<x>`. One line per `by` value;
    with agg_seeds, the line is the seed-mean and the band is +/-1 std."""
    xcol, ycol = f"al/{x}", f"al/{metric}"
    if ycol not in hist.columns:
        avail = sorted(c[3:] for c in hist.columns if c.startswith("al/"))
        raise KeyError(f"{ycol!r} not in history. Fetched metrics: {avail}")
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    keys = sorted(hist[by].dropna().unique(), key=str)
    cmap = plt.get_cmap("tab10" if len(keys) <= 10 else "tab20")
    for i, key in enumerate(keys):
        sub = hist[hist[by] == key]
        color = cmap(i % cmap.N)
        if agg_seeds:
            g = sub.dropna(subset=[ycol]).groupby(xcol)[ycol].agg(["mean", "std", "count"])
            ax.plot(g.index, g["mean"], marker="o", ms=3, color=color,
                    label=f"{key}  (n={int(g['count'].max())})")
            std = g["std"].fillna(0.0)
            ax.fill_between(g.index, g["mean"] - std, g["mean"] + std, color=color, alpha=0.15)
        else:
            for seed, s2 in sub.groupby("seed"):
                s2 = s2.sort_values(xcol)
                ax.plot(s2[xcol], s2[ycol], marker="o", ms=3, color=color,
                        alpha=0.7, label=f"{key}  seed {seed}")
    ax.set_xlabel(x); ax.set_ylabel(metric)
    ax.set_title(title or f"{metric} vs {x}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc="best")
    return ax

def plot_aubc_bars(runs, metric="test_acc", by="method", top=None, ax=None, figw=8):
    """Horizontal bar chart of AUBC (seed-mean +/- std) per `by` value."""
    tab = aubc_table(runs, metric, by=[by])
    if top:
        tab = tab.head(top)
    tab = tab.sort_values("mean")
    if ax is None:
        _, ax = plt.subplots(figsize=(figw, max(2.0, 0.42 * len(tab) + 1)))
    ax.barh(tab[by].astype(str), tab["mean"],
            xerr=tab["std"].fillna(0.0), capsize=3, color="#4C78A8")
    for y, (m, n) in enumerate(zip(tab["mean"], tab["n"])):
        ax.text(m, y, f"  {m:.3f} (n={int(n)})", va="center", fontsize=8)
    ax.set_xlabel(f"AUBC {metric}")
    ax.set_title(f"AUBC {metric} by {by}")
    ax.grid(True, axis="x", alpha=0.3)
    return ax

## 1. Load all runs

Reads from `SOURCE` (live wandb by default). `overview()` shows the datasets,
regimes, methods, and metrics you can filter on.

In [ ]:
runs = load_runs()
overview(runs)
runs.head()

## 2. Filter & group

Pick one dataset + regime, then rank methods by AUBC across seeds.
`method` is the full variant tag (kernel / norm / q / grad-vs-feat), so vendi
variants stay distinct. Filter examples:

```python
filter_runs(runs, data_name="cifar10_imb", active_name="cifar10_low", state="finished")
filter_runs(runs, data_name="mimic3_sand", query_name=["random", "bald", "badge"])
filter_runs(runs, method_contains="vendi-q2", tags_all=["kernel:cos"])
```

In [ ]:
sel = filter_runs(
    runs,
    data_name="cifar10_imb",
    active_name="cifar10_low",
    state="finished",
)
print(f"{len(sel)} runs, {sel['method'].nunique()} methods, "
      f"seeds {sorted(sel['seed'].dropna().unique())}")

aubc_table(sel, metric="test_acc").round(4)

## 3. Learning-curve plots

Fetch the per-iteration history for the filtered runs and plot `test_acc`
(mean over seeds, +/-1 std band) against the label budget `al/n_labelled`.
Fetch several metrics at once, e.g. `("test_acc", "test_auroc", "test_bacc")`.

In [ ]:
hist = load_history(sel, metrics=("test_acc",))    # add "test_auroc", "test_bacc", ...
print("history rows:", len(hist))

ax = plot_curves(hist, metric="test_acc", by="method")
ax.set_title("cifar10_imb / cifar10_low — test_acc vs budget")
plt.tight_layout(); plt.show()

### Focus on a few methods

Narrow to a handful and drop `agg_seeds` to see per-seed spread.

In [ ]:
focus = filter_runs(
    sel,
    method=["random", "bald", "badge", "entropy"],   # exact method names from overview()
)
# or: focus = sel[sel["method"].str.contains("vendi-q2-feat")]

hist_focus = load_history(focus, metrics=("test_acc",))
if len(hist_focus):
    plot_curves(hist_focus, metric="test_acc", by="method", agg_seeds=True)
    plt.tight_layout(); plt.show()
else:
    print("No matching runs — check method names in overview()/aubc_table().")

## 4. AUBC comparison bars

Single-number ranking of the filtered methods (seed-mean AUBC +/- std).

In [ ]:
plot_aubc_bars(sel, metric="test_acc", by="method")
plt.tight_layout(); plt.show()

## 5. Recipes

```python
# --- switch to the offline snapshot (no network) -------------------------
SOURCE = "analysis/wandb_snapshots/2026-09-21_pre-pheno-backfill"
runs = load_runs()

# --- compare the SAME method across regimes ------------------------------
sub = filter_runs(runs, data_name="cifar10_imb", method_contains="vendi-q2-feat-cos")
aubc_table(sub, "test_acc", by=["active_name", "method"])

# --- one metric, per-seed curves (no averaging) --------------------------
h = load_history(filter_runs(runs, data_name="mimic3_sand", active_name="mimic3_med",
                             query_name=["random", "badge", "bald"]),
                 metrics=("test_auprc",))
plot_curves(h, metric="test_auprc", by="query_name", agg_seeds=False)

# --- rank by a different AUBC metric -------------------------------------
aubc_table(sub, metric="test_auroc")        # or test_bacc / test_kappa / test_macro_f1 ...

# --- grid: several metrics side by side ----------------------------------
mets = ["test_acc", "test_auroc", "test_bacc"]
h = load_history(sel, metrics=mets)
fig, axes = plt.subplots(1, len(mets), figsize=(6*len(mets), 5))
for ax, m in zip(axes, mets):
    plot_curves(h, metric=m, by="method", ax=ax)
plt.tight_layout(); plt.show()

# --- jump to a run in the browser ----------------------------------------
sel[["name", "seed", "state", "url"]].head()
```